# 08 — Factor Discovery & ONNX Model Export

This notebook performs multi-factor analysis on TCPD Assembly Election data to:
1. Profile features across states and election cycles
2. Engineer prediction factors with coefficients
3. Train an XGBoost model for constituency-level predictions
4. Export the trained model as ONNX for API deployment
5. Generate SHAP explainability analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import shap
import json
import os

from db import engine

OUTPUT_DIR = '../output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Data Loading & Profiling

Load all post-delimitation assembly elections and profile the available features.

In [ ]:
# Load post-delimitation data (2008+) with all candidate-level features
df = pd.read_sql("""
    SELECT state_name, year, assembly_no, constituency_no, constituency_name,
           constituency_type, district_name, sub_region,
           position, candidate, sex, party, votes, age,
           candidate_type, valid_votes, electors,
           n_cand, turnout_percentage, vote_share_percentage,
           margin, margin_percentage, enop,
           incumbent, recontest, turncoat, same_constituency, same_party,
           no_terms, last_party, contested,
           election_type
    FROM tcpd_ae
    WHERE year >= 2008
      AND election_type = 'AE'
      AND (poll_no = 0 OR poll_no IS NULL)
    ORDER BY state_name, year, constituency_no, position
""", engine)

print(f"Loaded {len(df):,} candidate rows")
print(f"States: {df['state_name'].nunique()}")
print(f"Elections: {df.groupby(['state_name','year']).ngroups}")
print(f"\nFeature coverage:")
for col in ['incumbent','recontest','turncoat','same_constituency','same_party','no_terms','sex','age']:
    pct = (df[col].notna().sum() / len(df)) * 100
    print(f"  {col}: {pct:.1f}%")

## 2. Feature Engineering

Create prediction features at the constituency and candidate level.

In [ ]:
# ── Constituency-level features ──
# For each (state, year, constituency), compute:
# - winner's vote share, margin, turnout
# - ENOP, n_cand
# - whether constituency is reserved (SC/ST)
# - previous election's winner and margin (for incumbency/swing)

winners = df[df['position'] == 1].copy()
winners['is_reserved'] = (winners['constituency_type'] != 'GEN').astype(int)
winners['is_female'] = (winners['sex'] == 'F').astype(int)
winners['is_incumbent'] = (winners['incumbent'] == 'yes').astype(int)
winners['is_recontest'] = (winners['recontest'] == 'yes').astype(int)
winners['is_turncoat'] = (winners['turncoat'] == 'yes').astype(int)
winners['is_same_constituency'] = (winners['same_constituency'] == 'yes').astype(int)
winners['is_same_party'] = (winners['same_party'] == 'yes').astype(int)
winners['terms_served'] = pd.to_numeric(winners['no_terms'], errors='coerce').fillna(0)

# Get runner-up data for margin computation
runners = df[df['position'] == 2][['state_name','year','constituency_no','party','vote_share_percentage']].copy()
runners.columns = ['state_name','year','constituency_no','runner_up_party','runner_up_vote_share']

const_df = winners.merge(runners, on=['state_name','year','constituency_no'], how='left')
print(f"Constituency-elections: {len(const_df):,}")
const_df.head()

In [ ]:
# ── Previous election linkage ──
# Link each constituency to its previous election results

# Sort by state, constituency, year to enable shift operations
const_df = const_df.sort_values(['state_name','constituency_no','year'])

prev_cols = {
    'vote_share_percentage': 'prev_winner_vote_share',
    'margin_percentage': 'prev_margin_pct',
    'turnout_percentage': 'prev_turnout',
    'enop': 'prev_enop',
    'n_cand': 'prev_n_cand',
    'party': 'prev_winner_party',
}

for orig, new_name in prev_cols.items():
    const_df[new_name] = const_df.groupby(['state_name','constituency_no'])[orig].shift(1)

# Turnout change
const_df['turnout_change'] = const_df['turnout_percentage'] - const_df['prev_turnout']

# Party change (did the party flip?)
const_df['party_retained'] = (const_df['party'] == const_df['prev_winner_party']).astype(int)

# Drop first election per constituency (no previous data)
model_df = const_df.dropna(subset=['prev_margin_pct']).copy()
print(f"Constituency-elections with previous data: {len(model_df):,}")

## 3. Coefficient Derivation

Compute regression coefficients for each factor, both nationally and per state.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Target: did the incumbent party retain the seat?
# Features: the 12 factors

FEATURE_COLS = [
    'turnout_change',        # turnoutChange
    'is_incumbent',          # incumbencyFatigue (inverted)
    'is_turncoat',           # turncoatPenalty
    'is_recontest',          # recontestBonus
    'is_same_constituency',  # sameConstituencyBonus
    'prev_margin_pct',       # previousMarginFactor
    'prev_enop',             # enopFactor
    'prev_n_cand',           # nCandFactor
    'is_reserved',           # constituencyTypeFactor
    'is_female',             # genderFactor
    'terms_served',          # partyStrengthFactor (proxy)
    'prev_winner_vote_share' # partyVoteShareFactor
]

FACTOR_NAMES = [
    'turnoutChange', 'incumbencyFatigue', 'turncoatPenalty',
    'recontestBonus', 'sameConstituencyBonus', 'previousMarginFactor',
    'enopFactor', 'nCandFactor', 'constituencyTypeFactor',
    'genderFactor', 'partyStrengthFactor', 'partyVoteShareFactor'
]

def compute_coefficients(data, state_name='_national'):
    """Fit logistic regression and return coefficients."""
    subset = data.dropna(subset=FEATURE_COLS + ['party_retained'])
    if len(subset) < 50:
        return None
    
    X = subset[FEATURE_COLS].values
    y = subset['party_retained'].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    lr = LogisticRegression(max_iter=1000, C=1.0)
    lr.fit(X_scaled, y)
    
    # Convert coefficients to interpretable scale
    coeffs = {}
    for name, coef, std in zip(FACTOR_NAMES, lr.coef_[0], scaler.scale_):
        # Coefficient per unit of original feature
        coeffs[name] = round(float(coef / std), 4)
    
    score = lr.score(X_scaled, y)
    print(f"  {state_name}: accuracy={score:.3f}, n={len(subset)}")
    return coeffs

# National coefficients
print("Computing coefficients:")
all_coeffs = {}
national = compute_coefficients(model_df, '_national')
if national:
    all_coeffs['_national'] = national

# Per-state coefficients
for state in sorted(model_df['state_name'].unique()):
    state_data = model_df[model_df['state_name'] == state]
    coeffs = compute_coefficients(state_data, state.replace(' ', '_'))
    if coeffs:
        all_coeffs[state.replace(' ', '_')] = coeffs

print(f"\nGenerated coefficients for {len(all_coeffs)} state/national groups")

In [ ]:
# Export coefficients to JSON
with open(os.path.join(OUTPUT_DIR, 'coefficients.json'), 'w') as f:
    json.dump(all_coeffs, f, indent=2)

print("Exported coefficients.json")
print("\nNational coefficients:")
for k, v in all_coeffs.get('_national', {}).items():
    print(f"  {k}: {v:+.4f}")

## 4. XGBoost Model Training

Train an XGBoost classifier for constituency-level winner prediction.

In [ ]:
# Prepare data for XGBoost
# Target: party_retained (1 = incumbent party wins, 0 = seat flips)

train_df = model_df.dropna(subset=FEATURE_COLS + ['party_retained']).copy()

X = train_df[FEATURE_COLS].values
y = train_df['party_retained'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Class balance — Retained: {y.mean():.3f}, Flipped: {1-y.mean():.3f}")

In [ ]:
# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

# Evaluate
y_pred = model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Flipped','Retained']))

# Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"\nCV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

## 5. SHAP Explainability

Compute SHAP values to understand feature importance and interactions.

In [ ]:
# SHAP analysis
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary plot
shap.summary_plot(shap_values, X_test, feature_names=FACTOR_NAMES, show=False)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

# Feature importance from SHAP
shap_importance = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'factor': FACTOR_NAMES,
    'shap_importance': shap_importance
}).sort_values('shap_importance', ascending=False)

print("\nFeature Importance (SHAP):")
print(importance_df.to_string(index=False))

## 6. ONNX Export

Export the trained XGBoost model to ONNX format for API deployment.

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Define input shape
initial_type = [('features', FloatTensorType([None, len(FEATURE_COLS)]))]

# Convert to ONNX
onnx_model = convert_sklearn(
    model,
    initial_types=initial_type,
    target_opset=15,
    options={type(model): {'zipmap': False}},
)

# Add metadata
from onnx import helper
meta = onnx_model.metadata_props.add()
meta.key = 'version'
meta.value = '1.0.0'

meta = onnx_model.metadata_props.add()
meta.key = 'feature_names'
meta.value = json.dumps(FACTOR_NAMES)

meta = onnx_model.metadata_props.add()
meta.key = 'accuracy'
meta.value = str(round(accuracy_score(y_test, y_pred), 4))

# Save
onnx_path = os.path.join(OUTPUT_DIR, 'election_predictor.onnx')
with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())

print(f"Exported ONNX model to {onnx_path}")
print(f"Model size: {os.path.getsize(onnx_path) / 1024:.1f} KB")

In [ ]:
# Verify ONNX model
import onnxruntime as ort

sess = ort.InferenceSession(onnx_path)
input_name = sess.get_inputs()[0].name

# Test with a batch from test set
test_batch = X_test[:10].astype(np.float32)
onnx_pred = sess.run(None, {input_name: test_batch})

# Compare with sklearn predictions
sklearn_pred = model.predict(test_batch)
onnx_labels = onnx_pred[0]  # labels output

print("ONNX vs sklearn predictions match:", np.array_equal(sklearn_pred, onnx_labels))
print(f"ONNX output shapes: {[o.shape for o in onnx_pred]}")

# Print model metadata
meta = sess.get_modelmeta()
print(f"\nModel metadata:")
for k, v in meta.custom_metadata_map.items():
    print(f"  {k}: {v}")

## 7. Summary & Next Steps

Coefficient and model outputs are exported to `output/`:
- `coefficients.json` — per-state factor coefficients for the formula engine
- `election_predictor.onnx` — XGBoost model for API deployment
- `shap_summary.png` — feature importance visualization

To deploy the ONNX model:
1. Copy `election_predictor.onnx` to `api/models/`
2. The API will auto-load it via the `/predict/ml` endpoint
3. Update `frontend/src/engine/data/coefficients.json` with new coefficient values